# DEEP Game Subset Search — Exhaustive Combinatorial Optimisation

This notebook implements the **exhaustive combinatorial search** over all 2^14 − 1 = 16,383 non-empty subsets of the 14 DEEP games. For each subset, it evaluates how well the selected games' singleton latent scores predict GMDS subscales via 5-fold CV with an MLP regressor.

## Motivation

Running all 14 DEEP games takes significant field time. The goal is to find the **smallest subset of games** whose combined latent scores achieve criterion validity (GMDS prediction) comparable to the full battery. This makes DEEP faster and more practical for large-scale community screening.

## How the search works

1. **Pre-load singletons**: Load all 14 per-game latent CSVs from `singles/` (produced by `DEEP_scoring.ipynb`).
2. **Iterate subsets**: For each subset (min size 1, max size 14), combine the selected games' features by merging on `child_ids`.
3. **Evaluate**: Train an MLP + 5-fold CV → record mean Pearson r with GMDS subscales.
4. **Rank**: Sort all subsets by mean correlation (descending) and save results to CSV.

## Design decisions

- **Decomposition trick**: Training one VAE per game upfront (in `DEEP_scoring.ipynb`) avoids retraining the VAE for each of 16,383 subsets. Only the thin MLP head is re-trained per subset.
- **Parallelisation**: The search is distributed across CPU cores using `joblib.Parallel` (Loky backend). Each worker processes a batch of subsets (`SUBSET_BATCH_SIZE = 32`) to amortise overhead.
- **Incremental saving**: Results are flushed to CSV every `SAVE_EVERY = 512` subsets, so progress is preserved if the run is interrupted.

## Quick dry-run

Set `MAX_GAMES_PER_SUBSET = 3` to evaluate only subsets of size 1, 2, and 3 (364 subsets total) for a fast sanity check before the full exhaustive run.

## Key result

The top-ranked 5-game subset (`GYG`, `LR`, `OOO`, `SC`, `SR`) matches the full 14-game battery criterion validity on all five GMDS subscales (Table 7.1 of the thesis).

In [7]:
from itertools import combinations
from math import comb
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

# Ground-truth targets

gmds = pd.read_csv("gmds_scores.csv")
gmds = gmds[gmds["Age"]>=914]  # Exclude children under 2.5 years old
gmds.drop(columns=["Age"], inplace=True, errors="ignore")
target_cols = ["fol", "langCom", "eye_hand", "soc_per", "gross_mot"]

GAME_DIR = Path("singles")
GAME_FILES = sorted(GAME_DIR.glob("DEEP_Deep_abilities_*.csv"))


def extract_game_name(path):
    return Path(path).stem.replace("DEEP_Deep_abilities_", "").strip("_")


game_names = [extract_game_name(path) for path in GAME_FILES]

print(f"Loaded {len(game_names)} game files")
print(game_names)
print(f"GMDS rows: {len(gmds)}")

gmds.head()


Loaded 14 game files
['AT', 'GYG', 'HO', 'JIG', 'LR', 'MS', 'OOO', 'PB', 'PM', 'SC', 'SD', 'SO', 'SR', 'ST']
GMDS rows: 563


,child_ids,fol,langCom,eye_hand,soc_per,gross_mot
1,IN-0034-BL,45,52,52,61,50
2,IN-0055-BL,36,41,49,49,46
5,IN-0067-BL,51,51,60,52,52
6,IN-0077-BL,39,52,54,58,48
7,IN-0078-BL,29,30,36,31,36


## Data Loading

Loads GMDS scores (filtered to children aged ≥ 2.5 years, i.e., ≥ 914 days) and all 14 per-game singleton CSVs from the `singles/` directory. Game names are extracted from file names by stripping the `DEEP_Deep_abilities_` prefix.

## Search Infrastructure

The following functions implement the full subset evaluation and search machinery:

- **`load_game_feature_tables`**: Loads each singleton CSV, namespaces feature columns as `{game}__{col}` to avoid collisions.
- **`build_full_feature_dataframe`**: Left-joins all game tables onto the GMDS targets.
- **`build_subset_dataframe`**: Given a set of game names, constructs the feature matrix by selecting the appropriate namespaced columns and dropping rows with any missing target or feature.
- **`evaluate_model_per_target`**: 5-fold CV — clones the model, fits, predicts, and computes Pearson r / MSE / R² per GMDS subscale per fold.
- **`evaluate_subset`**: Calls `build_subset_dataframe` + `evaluate_model_per_target` and returns a summary dict plus per-target results.
- **`search_game_subsets`**: Outer loop — iterates all subsets in batches, dispatches to workers via `joblib.Parallel`, accumulates and saves results incrementally.

In [8]:
import os

from joblib import Parallel, cpu_count, delayed
from sklearn.base import clone
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits


def load_game_feature_tables(game_files):
    tables = {}
    feature_cols_by_game = {}

    for path in game_files:
        game_name = extract_game_name(path)
        game_df = pd.read_csv(path).drop(columns=["Age"], errors="ignore")
        feature_cols = [col for col in game_df.columns if col != "child_ids"]
        rename_map = {col: f"{game_name}__{col}" for col in feature_cols}
        renamed_df = game_df[["child_ids", *feature_cols]].rename(columns=rename_map)

        tables[game_name] = renamed_df
        feature_cols_by_game[game_name] = [col for col in renamed_df.columns if col != "child_ids"]

    return tables, feature_cols_by_game


game_feature_tables, game_feature_cols_by_game = load_game_feature_tables(GAME_FILES)


def build_full_feature_dataframe():
    full_df = gmds[["child_ids", *target_cols]].copy()

    for game_name in game_names:
        full_df = full_df.merge(game_feature_tables[game_name], on="child_ids", how="left")

    return full_df


full_feature_df = build_full_feature_dataframe()


def count_game_subsets(game_names, min_size=1, max_size=None):
    if max_size is None:
        max_size = len(game_names)
    return sum(comb(len(game_names), size) for size in range(min_size, max_size + 1))


def iter_game_subsets(game_names, min_size=1, max_size=None):
    if max_size is None:
        max_size = len(game_names)
    for size in range(min_size, max_size + 1):
        yield from combinations(game_names, size)


def get_feature_columns(selected_games):
    feature_cols = []

    for game_name in selected_games:
        feature_cols.extend(game_feature_cols_by_game[game_name])

    return feature_cols


def build_subset_dataframe(selected_games):
    feature_cols = get_feature_columns(selected_games)
    selected_cols = ["child_ids", *feature_cols, *target_cols]
    subset_df = full_feature_df[selected_cols].copy()

    subset_df = subset_df.dropna(subset=[*target_cols, *feature_cols]).reset_index(drop=True)
    return subset_df, feature_cols


sample_df, sample_feature_cols = build_subset_dataframe((game_names[0],))
print(sample_df.shape)
print(sample_feature_cols)
print(f"Full feature table shape: {full_feature_df.shape}")
print(f"Detected CPU cores: {os.cpu_count()}")


(559, 8)
['AT__accuracy', 'AT__total_DEEP']
Full feature table shape: (563, 54)
Detected CPU cores: 8


In [9]:
def evaluate_model_per_target(model, X, y, n_splits=5):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    fold_corrs = []
    fold_mses = []
    fold_r2s = []

    for train_idx, test_idx in kf.split(X):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        model_clone = clone(model)
        model_clone.fit(X_train, y_train)
        y_pred = model_clone.predict(X_test)

        y_pred = np.atleast_2d(y_pred)
        if y_pred.shape[0] != len(X_test):
            y_pred = y_pred.T

        y_true = y_test.values
        y_true = np.atleast_2d(y_true)
        if y_true.shape[0] != len(X_test):
            y_true = y_true.T

        corrs = []
        mses = []
        r2s = []

        for i in range(y_true.shape[1]):
            corr = pd.Series(y_pred[:, i]).corr(pd.Series(y_true[:, i]))
            mse = mean_squared_error(y_true[:, i], y_pred[:, i])
            r2 = r2_score(y_true[:, i], y_pred[:, i])

            corrs.append(corr)
            mses.append(mse)
            r2s.append(r2)

        fold_corrs.append(corrs)
        fold_mses.append(mses)
        fold_r2s.append(r2s)

    fold_corrs = np.array(fold_corrs)
    fold_mses = np.array(fold_mses)
    fold_r2s = np.array(fold_r2s)

    return pd.DataFrame({
        "target": y.columns,
        "target_mean_corr": fold_corrs.mean(axis=0),
        "target_std_corr": fold_corrs.std(axis=0),
        "target_mean_mse": fold_mses.mean(axis=0),
        "target_mean_r2": fold_r2s.mean(axis=0),
    })


def is_better_subset(current_summary, best_summary):
    if best_summary is None:
        return True

    current_key = (
        current_summary["mean_corr"],
        current_summary["mean_r2"],
        -current_summary["mean_mse"],
        -current_summary["subset_size"],
        current_summary["game_combo"],
    )
    best_key = (
        best_summary["mean_corr"],
        best_summary["mean_r2"],
        -best_summary["mean_mse"],
        -best_summary["subset_size"],
        best_summary["game_combo"],
    )
    return current_key > best_key


def evaluate_subset(model, selected_games, n_splits=5):
    subset_df, feature_cols = build_subset_dataframe(selected_games)
    X = subset_df[feature_cols]
    y = subset_df[target_cols]

    per_target_results = evaluate_model_per_target(model, X, y, n_splits=n_splits)
    summary = {
        "game_combo": "_".join(selected_games),
        "subset_size": len(selected_games),
        "num_rows": len(subset_df),
        "num_features": X.shape[1],
        "mean_corr": per_target_results["target_mean_corr"].mean(),
        "mean_std_corr": per_target_results["target_std_corr"].mean(),
        "mean_mse": per_target_results["target_mean_mse"].mean(),
        "mean_r2": per_target_results["target_mean_r2"].mean(),
    }
    return summary, per_target_results, subset_df


def build_detailed_results(summary, per_target_results):
    detailed_results = per_target_results.copy()
    detailed_results.insert(0, "game_combo", summary["game_combo"])
    detailed_results.insert(1, "subset_size", summary["subset_size"])
    detailed_results.insert(2, "num_rows", summary["num_rows"])
    detailed_results.insert(3, "num_features", summary["num_features"])
    detailed_results["combo_mean_corr"] = summary["mean_corr"]
    detailed_results["combo_mean_std_corr"] = summary["mean_std_corr"]
    detailed_results["combo_mean_mse"] = summary["mean_mse"]
    detailed_results["combo_mean_r2"] = summary["mean_r2"]
    detailed_results["target_order"] = detailed_results["target"].map(
        {target: idx for idx, target in enumerate(target_cols, start=1)}
    )
    return detailed_results


def finalize_summary_results(summary_df):
    summary_results_df = summary_df.sort_values(
        ["mean_corr", "mean_r2", "mean_mse", "subset_size", "game_combo"],
        ascending=[False, False, True, True, True],
    ).reset_index(drop=True)
    summary_results_df.insert(0, "combo_rank", np.arange(1, len(summary_results_df) + 1))
    return summary_results_df


def finalize_detailed_results(summary_results_df, detailed_df):
    detailed_results_df = detailed_df.merge(
        summary_results_df[["game_combo", "combo_rank"]],
        on="game_combo",
        how="left",
    )

    target_rank_df = detailed_results_df.sort_values(
        ["target", "target_mean_corr", "target_mean_r2", "target_mean_mse", "subset_size", "game_combo"],
        ascending=[True, False, False, True, True, True],
    )[["game_combo", "target"]].copy()
    target_rank_df["target_rank"] = target_rank_df.groupby("target").cumcount() + 1

    detailed_results_df = detailed_results_df.merge(
        target_rank_df,
        on=["game_combo", "target"],
        how="left",
    )

    detailed_results_df = detailed_results_df.sort_values(
        ["combo_rank", "target_order"],
        ascending=[True, True],
    ).reset_index(drop=True)

    ordered_columns = [
        "combo_rank",
        "target_rank",
        "game_combo",
        "subset_size",
        "target",
        "num_rows",
        "num_features",
        "target_mean_corr",
        "target_std_corr",
        "target_mean_mse",
        "target_mean_r2",
        "combo_mean_corr",
        "combo_mean_std_corr",
        "combo_mean_mse",
        "combo_mean_r2",
        "target_order",
    ]
    return detailed_results_df[ordered_columns]


def save_search_outputs(summary_records, detailed_records, summary_save_path=None, detail_save_path=None):
    summary_results_df = finalize_summary_results(pd.DataFrame(summary_records))
    detailed_results_df = finalize_detailed_results(
        summary_results_df,
        pd.concat(detailed_records, ignore_index=True),
    )

    if summary_save_path:
        summary_results_df.to_csv(summary_save_path, index=False)
    if detail_save_path:
        detailed_results_df.to_csv(detail_save_path, index=False)

    return summary_results_df, detailed_results_df


def chunk_sequence(sequence, chunk_size):
    for start in range(0, len(sequence), chunk_size):
        yield sequence[start:start + chunk_size]


def resolve_n_jobs(n_jobs):
    available = cpu_count()

    if n_jobs in (None, 0, 1):
        return 1
    if n_jobs == -1:
        return available
    if n_jobs < -1:
        return max(1, available + 1 + n_jobs)
    return min(n_jobs, available)


def evaluate_subset_batch(model, subset_batch, n_splits=5, inner_threads=1):
    batch_summary_records = []
    batch_detailed_records = []
    batch_best_result = None

    warnings.filterwarnings("ignore", category=ConvergenceWarning)

    with threadpool_limits(limits=inner_threads):
        for selected_games in subset_batch:
            summary, per_target_results, subset_df = evaluate_subset(
                model,
                selected_games,
                n_splits=n_splits,
            )
            batch_summary_records.append(summary)
            batch_detailed_records.append(build_detailed_results(summary, per_target_results))

            if is_better_subset(summary, None if batch_best_result is None else batch_best_result["summary"]):
                batch_best_result = {
                    "games": selected_games,
                    "summary": summary,
                    "per_target_results": per_target_results,
                    "subset_df": subset_df,
                }

    return {
        "summary_records": batch_summary_records,
        "detailed_results": pd.concat(batch_detailed_records, ignore_index=True),
        "best_result": batch_best_result,
        "num_processed": len(subset_batch),
    }


def search_game_subsets(
    model,
    game_names,
    min_size=1,
    max_size=None,
    n_splits=5,
    progress_every=100,
    summary_save_path=None,
    detail_save_path=None,
    save_every=250,
    n_jobs=1,
    subset_batch_size=32,
    inner_threads_per_worker=1,
):
    all_subsets = list(iter_game_subsets(game_names, min_size=min_size, max_size=max_size))
    total_subsets = len(all_subsets)
    effective_n_jobs = resolve_n_jobs(n_jobs)
    subset_batches = list(chunk_sequence(all_subsets, subset_batch_size))

    print(f"Evaluating {total_subsets} subsets using {effective_n_jobs} worker(s)")

    summary_records = []
    detailed_records = []
    best_result = None
    processed = 0
    last_progress_mark = 0
    last_save_mark = 0

    if effective_n_jobs == 1:
        batch_results = (
            evaluate_subset_batch(
                model,
                subset_batch,
                n_splits=n_splits,
                inner_threads=inner_threads_per_worker,
            )
            for subset_batch in subset_batches
        )
    else:
        batch_results = Parallel(
            n_jobs=effective_n_jobs,
            backend="loky",
            prefer="processes",
            return_as="generator_unordered",
        )(
            delayed(evaluate_subset_batch)(
                model,
                subset_batch,
                n_splits=n_splits,
                inner_threads=inner_threads_per_worker,
            )
            for subset_batch in subset_batches
        )

    for batch_result in batch_results:
        summary_records.extend(batch_result["summary_records"])
        detailed_records.append(batch_result["detailed_results"])
        processed += batch_result["num_processed"]

        batch_best_result = batch_result["best_result"]
        if is_better_subset(
            batch_best_result["summary"],
            None if best_result is None else best_result["summary"],
        ):
            best_result = batch_best_result

        if progress_every and (processed - last_progress_mark >= progress_every or processed == total_subsets):
            print(f"Processed {processed}/{total_subsets} subsets")
            last_progress_mark = processed

        if (
            (summary_save_path or detail_save_path)
            and save_every
            and (processed - last_save_mark >= save_every or processed == total_subsets)
        ):
            save_search_outputs(
                summary_records,
                detailed_records,
                summary_save_path=summary_save_path,
                detail_save_path=detail_save_path,
            )
            last_save_mark = processed

    summary_results_df, detailed_results_df = save_search_outputs(
        summary_records,
        detailed_records,
        summary_save_path=summary_save_path,
        detail_save_path=detail_save_path,
    )

    return summary_results_df, detailed_results_df, best_result


## Run the Search

Launches the exhaustive combinatorial search. Key configuration:

| Parameter | Default | Description |
|---|---|---|
| `MAX_GAMES_PER_SUBSET` | `None` | Maximum subset size; `None` = full exhaustive search (16,383 subsets) |
| `PARALLEL_JOBS` | `cpu_count − 1` | Number of parallel workers |
| `SUBSET_BATCH_SIZE` | 32 | Subsets per batch dispatched to each worker |
| `SAVE_EVERY` | 512 | Save progress every N subsets |
| `N_SPLITS` | 5 | Number of CV folds |

**Estimated runtime**: ~30–60 minutes for the full 16,383-subset search on an 8-core machine. Set `MAX_GAMES_PER_SUBSET = 3` for a quick 5-minute dry run (364 subsets).

In [10]:
from sklearn.neural_network import MLPRegressor

MIN_GAMES_PER_SUBSET = 1
MAX_GAMES_PER_SUBSET = None  # Set this to a smaller number, like 4, for a quicker dry run.
N_SPLITS = 5
PROGRESS_EVERY = 256
SAVE_EVERY = 512
PARALLEL_JOBS = max(1, min((cpu_count() or 1) - 1, 8))
SUBSET_BATCH_SIZE = 32
INNER_THREADS_PER_WORKER = 1
SUMMARY_RESULTS_OUTPUT_PATH = "mlp_subset_search_summary.csv"
DETAIL_RESULTS_OUTPUT_PATH = "mlp_subset_search_detailed.csv"

mlp_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", MLPRegressor(
        hidden_layer_sizes=(32,),
        alpha=1e-4,
        max_iter=2000,
        random_state=42,
    )),
])

search_summary_df, search_detailed_df, best_subset_result = search_game_subsets(
    model=mlp_model,
    game_names=game_names,
    min_size=MIN_GAMES_PER_SUBSET,
    max_size=MAX_GAMES_PER_SUBSET,
    n_splits=N_SPLITS,
    progress_every=PROGRESS_EVERY,
    summary_save_path=SUMMARY_RESULTS_OUTPUT_PATH,
    detail_save_path=DETAIL_RESULTS_OUTPUT_PATH,
    save_every=SAVE_EVERY,
    n_jobs=PARALLEL_JOBS,
    subset_batch_size=SUBSET_BATCH_SIZE,
    inner_threads_per_worker=INNER_THREADS_PER_WORKER,
)

df = best_subset_result["subset_df"].copy()
X = df.drop(columns=target_cols + ["child_ids"])
y = df[target_cols]

print("Sklearn MLP")
print("Best subset:", best_subset_result["games"])
print(f"Using {PARALLEL_JOBS} worker(s) with batch size {SUBSET_BATCH_SIZE}")
print(f"Saved combo summary results to {SUMMARY_RESULTS_OUTPUT_PATH}")
print(f"Saved per-target detailed results to {DETAIL_RESULTS_OUTPUT_PATH}")
print(f"Detailed rows: {len(search_detailed_df)}")

display(pd.DataFrame([best_subset_result["summary"]]).round(4))
display(search_summary_df.head(20).round(4))
display(search_detailed_df.head(25).round(4))
display(best_subset_result["per_target_results"].round(4))


Evaluating 16383 subsets using 7 worker(s)
Processed 256/16383 subsets
Processed 512/16383 subsets
Processed 768/16383 subsets
Processed 1024/16383 subsets
Processed 1280/16383 subsets
Processed 1536/16383 subsets
Processed 1792/16383 subsets
Processed 2048/16383 subsets
Processed 2304/16383 subsets
Processed 2560/16383 subsets
Processed 2816/16383 subsets
Processed 3072/16383 subsets
Processed 3328/16383 subsets
Processed 3584/16383 subsets
Processed 3840/16383 subsets
Processed 4096/16383 subsets
Processed 4352/16383 subsets
Processed 4608/16383 subsets
Processed 4864/16383 subsets
Processed 5120/16383 subsets
Processed 5376/16383 subsets
Processed 5632/16383 subsets
Processed 5888/16383 subsets
Processed 6144/16383 subsets
Processed 6400/16383 subsets
Processed 6656/16383 subsets
Processed 6912/16383 subsets
Processed 7168/16383 subsets
Processed 7424/16383 subsets
Processed 7680/16383 subsets
Processed 7936/16383 subsets
Processed 8192/16383 subsets
Processed 8448/16383 subsets
Pro

,game_combo,subset_size,num_rows,num_features,mean_corr,mean_std_corr,mean_mse,mean_r2
0,GYG_LR_OOO_SC_SR,5,559,18,0.7447,0.0393,29.0979,0.5393


,combo_rank,game_combo,subset_size,num_rows,num_features,mean_corr,mean_std_corr,mean_mse,mean_r2
0,1,GYG_LR_OOO_SC_SR,5,559,18,0.7447,0.0393,29.0979,0.5393
1,2,GYG_JIG_LR_SC_SR,5,559,18,0.7430,0.0349,28.9031,0.5423
2,3,GYG_LR_MS_PB_SC,5,559,16,0.7430,0.0337,29.0467,0.5397
3,4,GYG_LR_MS_OOO_SC_SO_SR,7,559,26,0.7424,0.0382,30.1689,0.5211
4,5,GYG_LR_MS_OOO_SR,5,559,18,0.7416,0.0409,29.3747,0.5350
5,6,GYG_LR_MS_OOO_SC,5,559,18,0.7414,0.0325,29.3957,0.5338
6,7,GYG_LR_PB_SC_SR,5,559,16,0.7407,0.0260,29.6229,0.5319
7,8,LR_MS_OOO_PB_SC_SO_SR,7,559,25,0.7399,0.0223,30.1328,0.5224
8,9,GYG_LR_MS_PB_SR,5,559,16,0.7398,0.0304,29.4881,0.5342
9,10,GYG_LR_MS_OOO_PB_SC,6,559,20,0.7392,0.0313,29.6062,0.5309


,combo_rank,target_rank,game_combo,subset_size,target,num_rows,num_features,target_mean_corr,target_std_corr,target_mean_mse,target_mean_r2,combo_mean_corr,combo_mean_std_corr,combo_mean_mse,combo_mean_r2,target_order
0,1,578,GYG_LR_OOO_SC_SR,5,fol,559,18,0.8091,0.0363,26.4425,0.6409,0.7447,0.0393,29.0979,0.5393,1
1,1,5,GYG_LR_OOO_SC_SR,5,langCom,559,18,0.7356,0.0457,26.4081,0.5193,0.7447,0.0393,29.0979,0.5393,2
2,1,28,GYG_LR_OOO_SC_SR,5,eye_hand,559,18,0.7745,0.0360,29.8062,0.5846,0.7447,0.0393,29.0979,0.5393,3
3,1,1,GYG_LR_OOO_SC_SR,5,soc_per,559,18,0.6887,0.0408,34.2143,0.4593,0.7447,0.0393,29.0979,0.5393,4
4,1,106,GYG_LR_OOO_SC_SR,5,gross_mot,559,18,0.7156,0.0378,28.6182,0.4925,0.7447,0.0393,29.0979,0.5393,5
5,2,29,GYG_JIG_LR_SC_SR,5,fol,559,18,0.8198,0.0237,25.0257,0.6617,0.7430,0.0349,28.9031,0.5423,1
6,2,28,GYG_JIG_LR_SC_SR,5,langCom,559,18,0.7276,0.0274,27.0081,0.5121,0.7430,0.0349,28.9031,0.5423,2
7,2,8,GYG_JIG_LR_SC_SR,5,eye_hand,559,18,0.7778,0.0392,29.0727,0.5955,0.7430,0.0349,28.9031,0.5423,3
8,2,22,GYG_JIG_LR_SC_SR,5,soc_per,559,18,0.6717,0.0462,35.4370,0.4391,0.7430,0.0349,28.9031,0.5423,4
9,2,72,GYG_JIG_LR_SC_SR,5,gross_mot,559,18,0.7182,0.0378,27.9721,0.5032,0.7430,0.0349,28.9031,0.5423,5


,target,target_mean_corr,target_std_corr,target_mean_mse,target_mean_r2
0,fol,0.8091,0.0363,26.4425,0.6409
1,langCom,0.7356,0.0457,26.4081,0.5193
2,eye_hand,0.7745,0.0360,29.8062,0.5846
3,soc_per,0.6887,0.0408,34.2143,0.4593
4,gross_mot,0.7156,0.0378,28.6182,0.4925


In [11]:
search_detailed_df.head(20)


,combo_rank,target_rank,game_combo,subset_size,target,num_rows,num_features,target_mean_corr,target_std_corr,target_mean_mse,target_mean_r2,combo_mean_corr,combo_mean_std_corr,combo_mean_mse,combo_mean_r2,target_order
0,1,578,GYG_LR_OOO_SC_SR,5,fol,559,18,0.809141,0.036274,26.442538,0.640922,0.744710,0.039326,29.097873,0.539336,1
1,1,5,GYG_LR_OOO_SC_SR,5,langCom,559,18,0.735625,0.045725,26.408087,0.519342,0.744710,0.039326,29.097873,0.539336,2
2,1,28,GYG_LR_OOO_SC_SR,5,eye_hand,559,18,0.774534,0.035985,29.806168,0.584601,0.744710,0.039326,29.097873,0.539336,3
3,1,1,GYG_LR_OOO_SC_SR,5,soc_per,559,18,0.688662,0.040823,34.214340,0.459328,0.744710,0.039326,29.097873,0.539336,4
4,1,106,GYG_LR_OOO_SC_SR,5,gross_mot,559,18,0.715589,0.037824,28.618232,0.492487,0.744710,0.039326,29.097873,0.539336,5
5,2,29,GYG_JIG_LR_SC_SR,5,fol,559,18,0.819791,0.023662,25.025706,0.661709,0.743037,0.034868,28.903121,0.542315,1
6,2,28,GYG_JIG_LR_SC_SR,5,langCom,559,18,0.727634,0.027440,27.008058,0.512093,0.743037,0.034868,28.903121,0.542315,2
7,2,8,GYG_JIG_LR_SC_SR,5,eye_hand,559,18,0.777823,0.039220,29.072679,0.595497,0.743037,0.034868,28.903121,0.542315,3
8,2,22,GYG_JIG_LR_SC_SR,5,soc_per,559,18,0.671717,0.046174,35.437029,0.439062,0.743037,0.034868,28.903121,0.542315,4
9,2,72,GYG_JIG_LR_SC_SR,5,gross_mot,559,18,0.718221,0.037844,27.972135,0.503216,0.743037,0.034868,28.903121,0.542315,5
